# ETL de movilidad urbana

El proceso construye viajes, movimientos de estación, perfiles anuales de usuario y contexto diario. Las ubicaciones conservan las coordenadas observadas y se enriquecen con territorio e infraestructura.

Ejecutar desde la raíz del caso o desde `notebooks/`, con las dependencias de `requirements.txt`. La lectura compartida está en `src/01_Fuentes.ipynb`. Las fuentes se verifican antes de procesar. La salida se prepara y comprueba antes de sustituir los archivos de `output/`.

Las fechas mantienen la hora declarada por el CSV. El enlace meteorológico usa la interpretación local de Buenos Aires como supuesto de trabajo documentado.

Las coordenadas 0,0 se identifican como no utilizables; los campos de posición de consumo quedan nulos y los valores originales permanecen en las fuentes.

In [1]:
from pathlib import Path
import gc, gzip, hashlib, io, json, os, tempfile
import numpy as np
import pandas as pd
from matplotlib.path import Path as Poligono
from pyproj import Geod, Transformer
import pyproj
from IPython.display import display, JSON

RAIZ=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/"catalogos/FUENTES.json").is_file() and (p/"input").is_dir()),None)
if RAIZ is None: raise FileNotFoundError("Inicia Jupyter desde la raíz del caso o notebooks/.")
carpeta_inicial = Path.cwd()
os.chdir(RAIZ)
%run src/01_Fuentes.ipynb
os.chdir(carpeta_inicial)
config = json.loads((RAIZ/"config/analisis.json").read_text(encoding="utf-8"))
anos = list(fuentes_viajes(RAIZ))
activacion_embudo = int(config["embudo"]["activacion_dias"])
retorno_embudo = int(config["embudo"]["retorno_dias"])
assert activacion_embudo in config["ventanas_activacion_dias"]
assert retorno_embudo in config["ventanas_retorno_dias"]
assert config["periodo_corredores"] == "anual"
assert config["umbral_lluvia_mm"] >= 0 and config["max_filas_particion_viajes"] > 0
resultado={"pyproj":pyproj.__version__, "parametros": config}
manifest=json.loads((RAIZ/"catalogos/FUENTES.json").read_text(encoding="utf-8"))
for fuente in manifest["fuentes"]:
    with (RAIZ/fuente["ruta"]).open("rb") as f:
        assert hashlib.file_digest(f,"sha256").hexdigest()==fuente["sha256"],fuente["ruta"]
print(f"{len(manifest['fuentes'])} fuentes y catálogos verificados.")

10 fuentes y catálogos verificados.


## 1. Lectura e identidad

Una réplica con contenido idéntico conserva la primera ocurrencia. Un ID con contenido contradictorio detiene la carga, para evitar una elección arbitraria. El ordinal de fuente identifica el registro de datos dentro del CSV, excluyendo su encabezado.

In [2]:
viajes,variantes,faltantes,esquemas,diccionarios=cargar_viajes(RAIZ)
validos,replica,conflicto=resolver_replicas(RAIZ,viajes)
if conflicto.any(): raise ValueError("Hay IDs con contenido contradictorio; revisa la fuente antes de exportar.")
assert viajes.Viaje.ge(0).all() and viajes.Usuario.ge(0).all()
brutos=len(viajes)
base=viajes.loc[validos].copy()
assert base.Viaje.is_unique
base["Fecha"]=base.Inicio.dt.normalize()
base["Hora"]=base.Inicio.dt.hour.astype("int8")
delta=(base.Fin-base.Inicio).dt.total_seconds()
observado=base.Inicio.notna()&base.Origen.ge(0)
completo=observado&base.Fin.notna()&delta.ge(0)&base.Destino.ge(0)
elegible=completo&base.Segundos.gt(0)
conciliado=elegible&delta.sub(base.Segundos).abs().le(1)
resultado["identidad"]={"brutos":brutos,"replicas_retiradas":int(replica.sum()),"viajes_unicos":len(base),
    "observados":int(observado.sum()),"completos":int(completo.sum()),"elegibles":int(elegible.sum()),"duracion_conciliada":int(conciliado.sum())}
display(pd.Series(resultado["identidad"]))
del viajes
_ = gc.collect()

2024: 1,000,000 registros leídos


2024: 2,000,000 registros leídos


2024: 3,000,000 registros leídos


2024: lectura completa, 3,559,284 registros


2025: 1,000,000 registros leídos


2025: 2,000,000 registros leídos


2025: 3,000,000 registros leídos


2025: lectura completa, 3,156,182 registros


Lectura terminada en 141 s; memoria compacta: 698 MB


brutos                 6715466
replicas_retiradas           1
viajes_unicos          6715465
observados             6715465
completos              6712085
elegibles              6677846
duracion_conciliada    6677830
dtype: int64

## 2. Ubicaciones observadas y nombres

La clave de ubicación combina ID interno de estación y coordenadas redondeadas a seis decimales, una resolución aproximada de 0,1 m. La clave 0 identifica una estación no determinada. Una estación identificada sin coordenadas conserva su propia fila sin ubicación.

El nombre estable procede del catálogo por ID interno; si el ID está ausente se utiliza el nombre observado dominante del último año disponible. Los nombres observados permanecen como alias. El número público es un atributo, no la clave de enlace.

In [3]:
estaciones=pd.read_csv(RAIZ/"catalogos/estaciones_bicicletas_fuente.csv")
assert estaciones.id.is_unique
def puntos_rol(rol):
    sufijo="O" if rol=="Origen" else "D"
    t=base[[rol,"Lon"+sufijo,"Lat"+sufijo]].copy()
    t.columns=["Estacion","Longitud","Latitud"]
    t[["Longitud","Latitud"]]=t[["Longitud","Latitud"]].round(6)
    geo=t.Longitud.between(-180,180)&t.Latitud.between(-90,90)
    t.loc[~geo,["Longitud","Latitud"]]=np.nan
    return t
origen=puntos_rol("Origen"); destino=puntos_rol("Destino")
ubicaciones=pd.concat([origen,destino],ignore_index=True).query("Estacion >= 0").drop_duplicates()
ubicaciones=ubicaciones.sort_values(["Estacion","Longitud","Latitud"],na_position="last").reset_index(drop=True)
ubicaciones.insert(0,"EstacionUbicacionKey",np.arange(1,len(ubicaciones)+1,dtype="int32"))
ubicaciones=pd.concat([pd.DataFrame([{"EstacionUbicacionKey":0,"Estacion":-1,"Longitud":np.nan,"Latitud":np.nan}]),ubicaciones],ignore_index=True)
nombre_catalogo=estaciones.set_index("id").nombre
nombre_observado=variantes.sort_values(["Ano","Eventos"],ascending=[False,False]).drop_duplicates("Estacion").set_index("Estacion").Nombre
ubicaciones["NombreEstacion"]=ubicaciones.Estacion.map(nombre_catalogo).fillna(ubicaciones.Estacion.map(nombre_observado)).fillna("Sin estación identificada")
ubicaciones["NumeroPublico"]=ubicaciones.Estacion.map(estaciones.set_index("id").numero_de_estacion).astype("Int64")
ubicaciones["EstacionKey"]=ubicaciones.Estacion.where(ubicaciones.Estacion.ge(0),0).astype("int32")
ubicaciones["EtiquetaEstacion"]=ubicaciones.EstacionKey.astype(str)+" · "+ubicaciones.NombreEstacion
aliases=variantes.groupby("Estacion").Nombre.agg(lambda s:" | ".join(sorted(set(s))))
ubicaciones["AliasObservados"]=ubicaciones.Estacion.map(aliases).fillna("Sin nombre observado")
assert not ubicaciones.duplicated(["Estacion","Longitud","Latitud"]).any()

## 3. Territorio y distancias métricas

El cruce con polígonos oficiales conserva huecos interiores y distingue asignación única, punto exterior y coordenada ausente. Las distancias a infraestructura son contexto de las instantáneas descargadas.

Se utiliza WGS84 / UTM zona 21S (EPSG:32721) para segmentos de ciclovía y geodesia WGS84 para puntos de subte. La proyección se contrasta con distancias geodésicas a esos puntos.

In [4]:
barrios=json.loads((RAIZ/"catalogos/barrios_fuente.geojson").read_text(encoding="utf-8"))
puntos=ubicaciones[["Longitud","Latitud"]].to_numpy(dtype=float)
cuenta=np.zeros(len(ubicaciones),dtype="int16")
ubicaciones["Barrio"]="Sin asignación"; ubicaciones["Comuna"]="Sin asignación"
for f in barrios["features"]:
    g=f["geometry"]; poligonos=[g["coordinates"]] if g["type"]=="Polygon" else g["coordinates"]
    dentro=np.zeros(len(ubicaciones),dtype=bool)
    for anillos in poligonos:
        pertence=Poligono(anillos[0]).contains_points(puntos)
        for hueco in anillos[1:]: pertence &= ~Poligono(hueco).contains_points(puntos)
        dentro |= pertence
    cuenta+=dentro
    ubicaciones.loc[dentro,"Barrio"]=f["properties"]["nombre"]
    ubicaciones.loc[dentro,"Comuna"]=str(f["properties"]["comuna"])
coordenada_cero=np.all(puntos==0,axis=1)
geo_disponible=np.isfinite(puntos).all(axis=1)&~coordenada_cero
ubicaciones["EstadoGeografia"]=np.select([~geo_disponible,cuenta==0,cuenta>1],
    ["Coordenada no disponible","Fuera de polígonos","Asignación ambigua"],default="Asignada")
ubicaciones.loc[coordenada_cero,"EstadoGeografia"]="Coordenada 0,0"
ubicaciones.loc[cuenta!=1,["Barrio","Comuna"]]="Sin asignación"
display(ubicaciones.EstadoGeografia.value_counts())

EstadoGeografia
Asignada                    435
Coordenada no disponible      9
Fuera de polígonos            2
Coordenada 0,0                1
Name: count, dtype: int64

In [5]:
proyeccion=Transformer.from_crs("EPSG:4326","EPSG:32721",always_xy=True)
geod=Geod(ellps="WGS84")
subte=json.loads((RAIZ/"catalogos/estaciones_subte_fuente.geojson").read_text(encoding="utf-8"))
nodos=np.array([f["geometry"]["coordinates"] for f in subte["features"]],dtype=float)
idx_geo=np.flatnonzero(geo_disponible)
p_geo=puntos[idx_geo]
xy=np.column_stack(proyeccion.transform(p_geo[:,0],p_geo[:,1]))
xy_subte=np.column_stack(proyeccion.transform(nodos[:,0],nodos[:,1]))
dist_geod=np.vstack([geod.inv(np.full(len(nodos),lon),np.full(len(nodos),lat),nodos[:,0],nodos[:,1])[2] for lon,lat in p_geo])
cercano=dist_geod.argmin(axis=1)
dist_subte=dist_geod[np.arange(len(idx_geo)),cercano]
dist_plana=np.linalg.norm(xy-xy_subte[cercano],axis=1)
error_relativo=np.abs(dist_plana-dist_subte)/np.maximum(dist_subte,1)
assert error_relativo.max()<.002
ubicaciones["DistanciaSubteActualM"]=np.nan
ubicaciones.loc[idx_geo,"DistanciaSubteActualM"]=dist_subte.round(1)
etiquetas_subte=[f['properties']['estacion'] for f in subte['features']]
ubicaciones["SubteActualMasCercano"]="Sin coordenada"
ubicaciones.loc[idx_geo,"SubteActualMasCercano"]=[etiquetas_subte[i] for i in cercano]
ciclovias=json.loads((RAIZ/"catalogos/ciclovias_fuente.geojson").read_text(encoding="utf-8"))
lineas=[np.array(f["geometry"]["coordinates"],dtype=float) for f in ciclovias["features"]]
lineas=[np.column_stack(proyeccion.transform(l[:,0],l[:,1])) for l in lineas]
a=np.concatenate([l[:-1] for l in lineas]); b=np.concatenate([l[1:] for l in lineas])
v=b-a; norma=(v*v).sum(axis=1)
dist_ciclo=[]
for p in xy:
    t=np.divide(((p-a)*v).sum(axis=1),norma,out=np.zeros_like(norma),where=norma>0).clip(0,1)
    dist_ciclo.append(float(np.linalg.norm(p-(a+t[:,None]*v),axis=1).min()))
ubicaciones["DistanciaCicloviaActualM"]=np.nan
ubicaciones.loc[idx_geo,"DistanciaCicloviaActualM"]=np.round(dist_ciclo,1)
resultado["geografia"]={"ubicaciones":len(ubicaciones),"con_coordenada":int(geo_disponible.sum()),
    "estado":ubicaciones.EstadoGeografia.value_counts().to_dict(),"error_relativo_utm_max":float(error_relativo.max())}
resultado["geografia"]["origenes_0_0"]=int((base.LonO.eq(0)&base.LatO.eq(0)).sum())
resultado["geografia"]["destinos_0_0"]=int((base.LonD.eq(0)&base.LatD.eq(0)).sum())
display(pd.Series(resultado["geografia"]))

ubicaciones                                                             447
con_coordenada                                                          437
estado                    {'Asignada': 435, 'Coordenada no disponible': ...
error_relativo_utm_max                                             0.000207
origenes_0_0                                                              0
destinos_0_0                                                            176
dtype: object

In [6]:
claves=ubicaciones[["Estacion","Longitud","Latitud","EstacionUbicacionKey"]]
def asignar_clave(puntos_rol):
    t=puntos_rol.merge(claves,on=["Estacion","Longitud","Latitud"],how="left",validate="many_to_one",sort=False)
    valor=t.EstacionUbicacionKey.where(t.Estacion.ge(0),0)
    assert valor.notna().all() and len(t)==len(base)
    return valor.to_numpy(dtype="int32")
base["UbicacionOrigenKey"]=asignar_clave(origen)
base["UbicacionDestinoKey"]=asignar_clave(destino)
pares=base[["UbicacionOrigenKey","UbicacionDestinoKey"]].drop_duplicates()
geo_por_clave=ubicaciones.set_index("EstacionUbicacionKey")[["Longitud","Latitud"]]
po=geo_por_clave.reindex(pares.UbicacionOrigenKey).to_numpy(dtype=float)
pdest=geo_por_clave.reindex(pares.UbicacionDestinoKey).to_numpy(dtype=float)
buena=np.isfinite(po).all(axis=1)&np.isfinite(pdest).all(axis=1)&~np.all(po==0,axis=1)&~np.all(pdest==0,axis=1)
pares["DistanciaDirectaKm"]=np.nan
pares.loc[buena,"DistanciaDirectaKm"]=(geod.inv(po[buena,0],po[buena,1],pdest[buena,0],pdest[buena,1])[2]/1000).round(6)
distancia=base[["UbicacionOrigenKey","UbicacionDestinoKey"]].merge(pares,how="left",validate="many_to_one",sort=False).DistanciaDirectaKm.to_numpy()
assert len(distancia)==len(base)
del origen,destino,pares
_ = gc.collect()

## 4. Usuarios y altas al mismo grano anual

La unión externa conserva usuarios con viajes y altas sin actividad. La clave usuario-año separa los periodos observados. Las altas aportan su fecha real y edad reportada; se utilizan fecha de alta, edad y género para el perfil anual.

La primera actividad usada en cohortes corresponde a viajes completos con duración positiva. Las medidas distinguen esta población de todas las salidas observadas.

In [7]:
altas=[]
for ano in anos:
    t=pd.read_csv(RAIZ/f"catalogos/usuarios_ecobici_{ano}.csv",dtype="string",encoding="utf-8-sig")
    t["Usuario"]=ids_enteros(t.iloc[:,0]); t["Ano"]=ano
    t["FechaAlta"]=pd.to_datetime(t.fecha_alta,errors="coerce")
    t["EdadReportada"]=pd.to_numeric(t.edad_usuario,errors="coerce").astype("Int64")
    t["GeneroAlta"]=t.genero_usuario.fillna("Sin dato").str.strip().replace("","Sin dato")
    altas.append(t[["Ano","Usuario","FechaAlta","EdadReportada","GeneroAlta"]])
altas=pd.concat(altas,ignore_index=True)
assert not altas.duplicated(["Ano","Usuario"]).any() and altas.Usuario.ge(0).all()
universo=pd.concat([base[["Ano","Usuario"]],altas[["Ano","Usuario"]]],ignore_index=True).drop_duplicates().sort_values(["Ano","Usuario"])
usuarios=universo.set_index(["Ano","Usuario"])
usuarios["UsuarioAnoKey"]=np.arange(1,len(usuarios)+1,dtype="int32")
actividad=base.loc[elegible].groupby(["Ano","Usuario"]).agg(FechaPrimeraObservada=("Fecha","min"),
    FechaUltimaObservada=("Fecha","max"),ViajesElegibles=("Viaje","size"),DiasActivos=("Fecha","nunique"),EstacionesOrigenDistintas=("Origen","nunique"))
usuarios=usuarios.join(actividad,validate="one_to_one").join(altas.set_index(["Ano","Usuario"]),validate="one_to_one")
conteo_observados=base.loc[observado].groupby(["Ano","Usuario"]).size().rename("ViajesObservados")
usuarios=usuarios.join(conteo_observados,validate="one_to_one")
for c in ("ViajesObservados","ViajesElegibles","DiasActivos","EstacionesOrigenDistintas"):
    usuarios[c]=usuarios[c].fillna(0).astype("int32")
usuarios["EsAltaObservada"]=usuarios.FechaAlta.notna().astype("int8")
usuarios["TieneViajeObservado"]=usuarios.ViajesObservados.gt(0).astype("int8")
usuarios["TieneViajeElegible"]=usuarios.ViajesElegibles.gt(0).astype("int8")
cierres=base.groupby("Ano").Fecha.max()
usuarios["FechaCorte"]=usuarios.index.get_level_values("Ano").map(cierres)
usuarios["RecenciaCierreDias"]=(usuarios.FechaCorte-usuarios.FechaUltimaObservada).dt.days.astype("Int64")
usuarios["SeguimientoObservadoDias"]=(usuarios.FechaCorte-usuarios.FechaPrimeraObservada).dt.days.astype("Int64")
usuarios["DiasHastaActivacion"]=(usuarios.FechaPrimeraObservada-usuarios.FechaAlta).dt.days.astype("Int64")
usuarios["EsCronologiaContradictoria"]=usuarios.DiasHastaActivacion.lt(0).fillna(False).astype("int8")

In [8]:
dias=base.loc[elegible,["Ano","Usuario","Fecha"]].drop_duplicates().join(usuarios.FechaPrimeraObservada,on=["Ano","Usuario"],validate="many_to_one")
dias["DiasDesdePrimera"]=(dias.Fecha-dias.FechaPrimeraObservada).dt.days
for ventana in config["ventanas_retorno_dias"]:
    tiene_retorno=dias.loc[dias.DiasDesdePrimera.between(1,ventana)].groupby(["Ano","Usuario"]).size().gt(0).reindex(usuarios.index,fill_value=False)
    elegibilidad=usuarios.SeguimientoObservadoDias.ge(ventana).fillna(False)
    usuarios[f"ElegibleRetorno{ventana}"]=elegibilidad.astype("int8")
    usuarios[f"Retorno{ventana}"]=tiene_retorno.where(elegibilidad,pd.NA).astype("Int8")
for ventana in config["ventanas_activacion_dias"]:
    elegibilidad=usuarios.FechaAlta.notna()&(usuarios.FechaCorte-usuarios.FechaAlta).dt.days.ge(ventana)&usuarios.EsCronologiaContradictoria.eq(0)
    activada=usuarios.DiasHastaActivacion.between(0,ventana).fillna(False)
    usuarios[f"ElegibleActivacion{ventana}"]=elegibilidad.astype("int8")
    usuarios[f"Activacion{ventana}"]=activada.where(elegibilidad,pd.NA).astype("Int8")
usuarios["EdadAnalitica"]=usuarios.EdadReportada.where(usuarios.EdadReportada.between(0,120)).astype("Int64")
usuarios["GrupoEdad"]=pd.cut(usuarios.EdadAnalitica.astype("float64"),[-1,17,24,34,44,54,64,120],
    labels=["0–17","18–24","25–34","35–44","45–54","55–64","65–120"]).astype("string").fillna("Sin edad utilizable")
usuarios["GeneroAlta"]=usuarios.GeneroAlta.fillna("Sin alta en catálogo anual")
usuarios["CohorteObservada"]=usuarios.FechaPrimeraObservada.dt.to_period("M").dt.to_timestamp()
usuarios["CohorteAlta"]=usuarios.FechaAlta.dt.to_period("M").dt.to_timestamp()
usuarios["GrupoFrecuencia"]=pd.cut(usuarios.ViajesElegibles,[-1,0,1,5,20,np.inf],
    labels=["Sin viaje elegible","1 viaje","2–5 viajes","6–20 viajes","Más de 20 viajes"]).astype("string")
usuarios["EsElegibleEmbudo"] = (usuarios.EsAltaObservada.eq(1) &
    usuarios.FechaAlta.le(usuarios.FechaCorte-pd.to_timedelta(activacion_embudo+retorno_embudo, unit="D"))).astype("int8")
indice_usuario=pd.MultiIndex.from_frame(base[["Ano","Usuario"]])
base["UsuarioAnoKey"]=usuarios.UsuarioAnoKey.reindex(indice_usuario).to_numpy(dtype="int32")
fact_usuarios=usuarios.reset_index().drop(columns="Usuario")
del dias
_ = gc.collect()
display(fact_usuarios.groupby("Ano")[["EsAltaObservada","TieneViajeElegible","ViajesElegibles"]].sum())

,EsAltaObservada,TieneViajeElegible,ViajesElegibles
Ano,,,
2024,197077,261432,3521665
2025,192674,286322,3156181


## 5. Viajes y movimientos de estación

Las banderas separan observación, destino completo, viaje elegible y duración conciliada. Los extremos permanecen identificados. La distancia directa es geodésica entre extremos y la razón distancia/tiempo se usa exclusivamente como diagnóstico.

Las llegadas se sitúan en la fecha y hora de destino. La suma de salidas y la suma de llegadas deben coincidir, por separado, con el número de viajes elegibles.

In [9]:
fact_viajes=pd.DataFrame({"ViajeKey":base.Viaje,"Ano":base.Ano,"FilaFuente":base.FilaFuente,
    "UsuarioAnoKey":base.UsuarioAnoKey,"FechaOrigen":base.Fecha,"HoraOrigen":base.Hora,
    "FechaDestino":base.Fin.dt.normalize(),"HoraDestino":base.Fin.dt.hour.astype("Int8"),
    "EstacionOrigenUbicacionKey":base.UbicacionOrigenKey,"EstacionDestinoUbicacionKey":base.UbicacionDestinoKey,
    "DuracionDeclaradaSegundos":base.Segundos.astype("Int64"),
    "DuracionAnaliticaSegundos":delta.where(conciliado).astype("Int64"),"DistanciaDirectaKm":distancia})
for campo in ("Modelo","Genero"):
    fact_viajes[campo+"Reportado"]=base[campo].map({v:k for k,v in diccionarios[campo].items()})
for nombre,mascara in {"EsObservado":observado,"EsCompleto":completo,"EsViajeElegible":elegible,
    "EsDuracionConciliada":conciliado,"EsDuracionNoPositiva":base.Segundos.le(0),
    "EsDuracionMayor24h":base.Segundos.gt(86400),"EsMismaEstacion":base.Origen.eq(base.Destino)&base.Origen.ge(0)}.items():
    fact_viajes[nombre]=mascara.astype("int8")
razon=np.divide(distancia,base.Segundos.to_numpy()/3600,out=np.full(len(base),np.nan),where=base.Segundos.to_numpy()>0)
fact_viajes["EsRazonDistanciaTiempoAlta"]=(elegible.to_numpy()&(razon>60)).astype("int8")
salidas=base.loc[elegible].groupby(["UbicacionOrigenKey","Fecha","Hora"]).size().rename("Salidas")
llegadas=base.loc[elegible].groupby(["UbicacionDestinoKey",base.Fin.dt.normalize(),base.Fin.dt.hour]).size().rename("Llegadas")
salidas.index.names=llegadas.index.names=["EstacionUbicacionKey","Fecha","Hora"]
fact_movimientos=pd.concat([salidas,llegadas],axis=1).fillna(0).astype("int32").reset_index()
fact_movimientos["Hora"]=fact_movimientos.Hora.astype("int8")
fact_movimientos["Saldo"]=fact_movimientos.Llegadas-fact_movimientos.Salidas
assert fact_movimientos.Salidas.sum()==fact_movimientos.Llegadas.sum()==elegible.sum()
dim_ubicaciones=ubicaciones.drop(columns="Estacion").copy()
dim_ubicaciones.loc[coordenada_cero,["Longitud","Latitud"]]=np.nan
assert dim_ubicaciones.EstacionUbicacionKey.is_unique
assert not fact_movimientos.duplicated(["EstacionUbicacionKey","Fecha","Hora"]).any()
display(pd.Series({"Viajes":len(fact_viajes),"Movimientos agregados":len(fact_movimientos),"Usuario-año":len(fact_usuarios),"Ubicaciones":len(dim_ubicaciones)}))

Viajes                   6715465
Movimientos agregados    3657462
Usuario-año               659392
Ubicaciones                  447
dtype: int64

### Conexiones entre ubicaciones

Una conexión reúne viajes elegibles por año, origen y destino observados. Se conservan todos los pares representables; el límite de conexiones visibles se aplica en el reporte. Los viajes circulares o sin geografía utilizable permanecen en el detalle y se concilian por separado.

In [10]:
campos_corredor = ["Ano", "EstacionOrigenUbicacionKey", "EstacionDestinoUbicacionKey"]
corredores = (fact_viajes.loc[fact_viajes.EsViajeElegible.eq(1)]
    .groupby(campos_corredor).size().rename("Viajes").reset_index())
assert corredores.Viajes.sum() == elegible.sum()
geo = dim_ubicaciones[["EstacionUbicacionKey", "EstacionKey", "NombreEstacion", "Latitud", "Longitud"]]
for rol, sufijo in (("Origen", "O"), ("Destino", "D")):
    referencia = geo.rename(columns={"EstacionUbicacionKey":f"Estacion{rol}UbicacionKey",
        "EstacionKey":f"Id{sufijo}", "NombreEstacion":rol, "Latitud":f"Lat{sufijo}", "Longitud":f"Lon{sufijo}"})
    corredores = corredores.merge(referencia, on=f"Estacion{rol}UbicacionKey", validate="many_to_one")
utilizables = corredores[["LatO","LonO","LatD","LonD"]].notna().all(axis=1) & corredores.IdO.ne(corredores.IdD)
resultado["corredores"] = {"viajes_totales":int(corredores.Viajes.sum()),
    "viajes_representables":int(corredores.loc[utilizables,"Viajes"].sum()),
    "viajes_circulares_o_sin_geografia":int(corredores.loc[~utilizables,"Viajes"].sum())}
corredores = corredores.loc[utilizables].sort_values(campos_corredor).reset_index(drop=True)
corredores["RutaKey"] = np.arange(1,len(corredores)+1,dtype="int64")
corredores["Ruta"] = corredores.Origen+" → "+corredores.Destino
extremos = []
for orden, sufijo in enumerate(("O","D")):
    extremos.append(corredores.assign(Orden=orden, Latitud=corredores[f"Lat{sufijo}"], Longitud=corredores[f"Lon{sufijo}"]))
columnas = ["Ano","RutaKey","Ruta","Orden","Latitud","Longitud","Viajes","Origen","Destino"]+campos_corredor[1:]
puntos_corredor = pd.concat(extremos,ignore_index=True)[columnas].sort_values(["RutaKey","Orden"])
assert not puntos_corredor.duplicated(["RutaKey","Orden"]).any()
assert puntos_corredor.groupby("RutaKey").Orden.agg(set).eq({0,1}).all()
assert puntos_corredor.loc[puntos_corredor.Orden.eq(0),"Viajes"].sum() == resultado["corredores"]["viajes_representables"]
resultado["corredores"].update(pares=len(corredores), puntos=len(puntos_corredor))
display(pd.Series(resultado["corredores"]))


viajes_totales                       6677846
viajes_representables                5928775
viajes_circulares_o_sin_geografia     749071
pares                                 190300
puntos                                380600
dtype: int64

In [11]:
directorio_actual = Path.cwd()
os.chdir(RAIZ)
%run src/02_Geografia_Visual.ipynb
os.chdir(directorio_actual)
geografia_visual = json.loads((RAIZ/"catalogos/barrios_fuente.geojson").read_text(encoding="utf-8"))
puntos_corredor, base_svg = enriquecer_mapa_local(puntos_corredor, geografia_visual)
(RAIZ/"assets").mkdir(exist_ok=True)

assert puntos_corredor.loc[puntos_corredor.Orden.eq(0),"TrazoSVG"].str.len().gt(0).all()
display(pd.Series({"Puntos":len(puntos_corredor),"Caracteres_base_svg":len(base_svg)}))

Puntos                  380600
Caracteres_base_svg      11280
dtype: int64

## 6. Contexto diario

El calendario distingue feriado nacional y día no laborable de aplicación general. El clima corresponde a una celda ERA5 y conserva sus unidades: °C, mm y km/h. La fecha de referencia se valida antes de unir, sin multiplicar registros.

In [12]:
clima_fuente=json.loads((RAIZ/"catalogos/clima_diario_era5_fuente.json").read_text(encoding="utf-8"))
contexto=pd.DataFrame(clima_fuente["daily"]).rename(columns={"time":"Fecha","temperature_2m_mean":"TemperaturaMediaC",
    "precipitation_sum":"PrecipitacionMm","wind_speed_10m_max":"VientoMaximoKmh"})
contexto["Fecha"]=pd.to_datetime(contexto.Fecha)
calendario=pd.read_csv(RAIZ/"catalogos/calendario_especial_2024_2025.csv",parse_dates=["Fecha"])
assert contexto.Fecha.is_unique and calendario.Fecha.is_unique
fact_contexto=contexto.merge(calendario[["Fecha","Tipo","Nombre"]],on="Fecha",how="left",validate="one_to_one").rename(columns={"Tipo":"TipoDiaEspecial","Nombre":"NombreDiaEspecial"})
fact_contexto[["TipoDiaEspecial","NombreDiaEspecial"]]=fact_contexto[["TipoDiaEspecial","NombreDiaEspecial"]].fillna("Sin fecha especial general")
fact_contexto["LatitudReferencia"]=clima_fuente["latitude"]
fact_contexto["LongitudReferencia"]=clima_fuente["longitude"]
fact_contexto["ModeloClima"]="ERA5"
fact_contexto["TipoClima"] = np.where(fact_contexto.PrecipitacionMm.ge(config["umbral_lluvia_mm"]),
    "Lluvia ≥"+str(config["umbral_lluvia_mm"])+" mm", "Menos de "+str(config["umbral_lluvia_mm"])+" mm")
assert not contexto.isna().any().any()
assert pd.Index(base.Fecha.unique()).difference(fact_contexto.Fecha).empty
resultado["contexto"]={"dias":len(fact_contexto),"fechas_especiales":len(calendario)}

## 7. Exportación determinista y sustitución controlada

Los CSV grandes se guardan en GZip, con fecha de compresión fija y sin ruta de origen en la cabecera. Los viajes se particionan por año. Las siete salidas se importan con Power Query; las comprimidas se leen mediante `Binary.Decompress` para `.gz`.

La carpeta temporal pertenece al caso y se retira al terminar. Si `output/` contiene archivos ajenos al contrato se detiene la exportación para conservarlos. Una reejecución sustituye el conjunto completo; no agrega filas al resultado anterior.

In [13]:
tablas = {}
max_filas = int(config["max_filas_particion_viajes"])
for ano, anual in fact_viajes.groupby("Ano", sort=True):
    anual = anual.sort_values("ViajeKey")
    partes = (len(anual)+max_filas-1)//max_filas
    for parte, inicio in enumerate(range(0, len(anual), max_filas), 1):
        sufijo = f"_parte{parte:03d}" if partes > 1 else ""
        tablas[f"FactViajes_{ano}{sufijo}.csv.gz"] = anual.iloc[inicio:inicio+max_filas]
tablas.update({"PuntosCorredor.csv":puntos_corredor,
    "FactMovimientosEstacion.csv.gz":fact_movimientos.sort_values(["EstacionUbicacionKey","Fecha","Hora"]),
    "FactUsuarioAno.csv.gz":fact_usuarios.sort_values("UsuarioAnoKey"),
    "FactContextoDiario.csv":fact_contexto.sort_values("Fecha"),
    "DimEstacionUbicacion.csv":dim_ubicaciones.sort_values("EstacionUbicacionKey")})
salida=RAIZ/"output"; salida.mkdir(exist_ok=True)
particiones_previas = {p.name for p in salida.glob("FactViajes_*.csv.gz")}
ajenos={p.name for p in salida.iterdir()}-set(tablas)-particiones_previas
if ajenos: raise ValueError(f"Archivos ajenos al contrato de output: {sorted(ajenos)}")
temporal=tempfile.TemporaryDirectory(prefix=".etl_",dir=RAIZ)
preparada=Path(temporal.name)
(preparada/"mapa_base_corredores.svg").write_text(base_svg,encoding="utf-8")
def escribir_csv(tabla,ruta):
    if ruta.suffix==".gz":
        with ruta.open("wb") as raw:
            with gzip.GzipFile(filename="",mode="wb",fileobj=raw,mtime=0,compresslevel=6) as gz:
                with io.TextIOWrapper(gz,encoding="utf-8",newline="") as texto:
                    tabla.to_csv(texto,index=False,date_format="%Y-%m-%d",lineterminator="\n",float_format="%.6f",chunksize=100_000)
    else:
        tabla.to_csv(ruta,index=False,encoding="utf-8",date_format="%Y-%m-%d",lineterminator="\n",float_format="%.6f")
for nombre,tabla in tablas.items():
    escribir_csv(tabla,preparada/nombre)
    assert (preparada/nombre).stat().st_size<100*1024*1024,nombre
    print(f"{nombre}: {len(tabla):,} filas",flush=True)

FactViajes_2024.csv.gz: 3,559,283 filas


FactViajes_2025.csv.gz: 3,156,182 filas


PuntosCorredor.csv: 380,600 filas


FactMovimientosEstacion.csv.gz: 3,657,462 filas


FactUsuarioAno.csv.gz: 659,392 filas


FactContextoDiario.csv: 731 filas


DimEstacionUbicacion.csv: 447 filas


## 8. Lectura de retorno y conciliaciones

Se comprueban las filas y claves de todos los archivos exportados, las referencias a usuario y ubicación, las poblaciones, las tasas y los nulos de las ventanas incompletas. La prueba vuelve a serializar el conjunto completo y compara sus huellas para verificar determinismo.

In [14]:
cuentas={}; claves_leidas=[]; suma_elegibles=0; suma_observados=0; suma_duracion=0
ids_u=set(fact_usuarios.UsuarioAnoKey); ids_geo=set(dim_ubicaciones.EstacionUbicacionKey)
for nombre,tabla in tablas.items():
    leidas=0
    for bloque in pd.read_csv(preparada/nombre,chunksize=200_000,low_memory=False):
        leidas+=len(bloque)
        if nombre=="DimEstacionUbicacion.csv":
            assert not (bloque.Longitud.eq(0)&bloque.Latitud.eq(0)).any()
        if nombre.startswith("FactViajes_"):
            assert bloque.UsuarioAnoKey.isin(ids_u).all()
            assert bloque.EstacionOrigenUbicacionKey.isin(ids_geo).all() and bloque.EstacionDestinoUbicacionKey.isin(ids_geo).all()
            assert pd.to_datetime(bloque.FechaOrigen,errors="coerce").notna().all()
            assert bloque.loc[bloque.EsDuracionConciliada.eq(0),"DuracionAnaliticaSegundos"].isna().all()
            claves_leidas.append(bloque.ViajeKey.to_numpy(dtype="int64"))
            suma_elegibles+=int(bloque.EsViajeElegible.sum()); suma_observados+=int(bloque.EsObservado.sum())
            suma_duracion+=int(bloque.DuracionAnaliticaSegundos.sum())
        if nombre=="FactUsuarioAno.csv.gz":
            for prefijo,ventanas in (("Retorno",config["ventanas_retorno_dias"]),("Activacion",config["ventanas_activacion_dias"])):
                for v in ventanas:
                    assert bloque.loc[bloque[f"Elegible{prefijo}{v}"].eq(0),f"{prefijo}{v}"].isna().all()
                    assert bloque[f"{prefijo}{v}"].dropna().isin([0,1]).all()
        if nombre=="FactMovimientosEstacion.csv.gz": assert bloque.EstacionUbicacionKey.isin(ids_geo).all()
    assert leidas==len(tabla),nombre
    cuentas[nombre]=leidas
assert np.array_equal(np.sort(np.concatenate(claves_leidas)),np.sort(base.Viaje.to_numpy()))
assert suma_elegibles==int(elegible.sum()) and suma_observados==int(observado.sum())
assert suma_duracion==int(fact_viajes.DuracionAnaliticaSegundos.sum())
assert fact_usuarios.ViajesElegibles.sum()==elegible.sum()
resultado["filas_exportadas"]=cuentas
resultado["conciliacion"]={"viajes_elegibles":suma_elegibles,"salidas":int(fact_movimientos.Salidas.sum()),
    "llegadas":int(fact_movimientos.Llegadas.sum()),"viajes_por_usuario":int(fact_usuarios.ViajesElegibles.sum()),"duracion_analitica_segundos":suma_duracion}
display(pd.Series(cuentas,name="Filas comprobadas"))

FactViajes_2024.csv.gz            3559283
FactViajes_2025.csv.gz            3156182
PuntosCorredor.csv                 380600
FactMovimientosEstacion.csv.gz    3657462
FactUsuarioAno.csv.gz              659392
FactContextoDiario.csv                731
DimEstacionUbicacion.csv              447
Name: Filas comprobadas, dtype: int64

In [15]:
huellas={}
for nombre,tabla in tablas.items():
    ruta=preparada/nombre
    with ruta.open("rb") as f: huella=hashlib.file_digest(f,"sha256").hexdigest()
    repetida=preparada/("repetida_"+nombre)
    escribir_csv(tabla,repetida)
    with repetida.open("rb") as f: assert hashlib.file_digest(f,"sha256").hexdigest()==huella,nombre
    repetida.unlink()
    huellas[nombre]=huella
for nombre in tablas: os.replace(preparada/nombre,salida/nombre)
for nombre in particiones_previas-set(tablas):
    (salida/nombre).unlink()
os.replace(preparada/"mapa_base_corredores.svg",RAIZ/"assets/mapa_base_corredores.svg")
temporal.cleanup()
resultado["sha256"]=huellas
resultado["controles"]={"identidad_unica":bool(base.Viaje.is_unique),"integridad_referencial":True,
    "lectura_retorno":True,"ventanas_incompletas_nulas":True,"conciliacion":True,"serializacion_repetida_identica":True,
    "sin_temporales_output":bool({p.name for p in salida.iterdir()}==set(tablas))}
resultado["archivos_mb"]={p.name:round(p.stat().st_size/1e6,2) for p in salida.iterdir()}
resultado["activacion30"]={str(int(ano)):{"elegibles":int(g.ElegibleActivacion30.sum()),"activadas":int(g.Activacion30.sum())} for ano,g in fact_usuarios.groupby("Ano")}
resultado["retorno30"]={str(int(ano)):{"elegibles":int(g.ElegibleRetorno30.sum()),"retornaron":int(g.Retorno30.sum())} for ano,g in fact_usuarios.groupby("Ano")}
display(pd.Series(resultado["controles"]))
display(JSON(json.loads(json.dumps(resultado,default=lambda x:x.item() if hasattr(x,"item") else str(x)))))

identidad_unica                    True
integridad_referencial             True
lectura_retorno                    True
ventanas_incompletas_nulas         True
conciliacion                       True
serializacion_repetida_identica    True
sin_temporales_output              True
dtype: bool

<IPython.core.display.JSON object>